# Линейная регрессия, градиентный спуск и регуляризация

На синтетических данных можно управлять шумом, масштабом и числом признаков, а найденные коэффициенты — сравнивать с известной зависимостью.

## Содержание

1. Линейная модель, функция потерь и максимальное правдоподобие.
2. Аналитическое решение методом наименьших квадратов.
3. Градиентный спуск.
4. Масштабирование признаков и mini-batch.
5. Регуляризация: Ridge и Lasso.
6. Модель, критерий и алгоритм обучения.
7. Задания.
8. Основные выводы.

## 0. Подготовка
Запустите ячейки сверху вниз. Демонстрации полностью работают без интернета и внешних данных.

Нужны `numpy`, `matplotlib`, `scipy`, `scikit-learn`, `IPython`, `threadpoolctl` — они доступны в среде курса. `ipywidgets` необязателен: без него остаются обычные графики, а параметры можно менять в вызовах функций. Для ползунков при необходимости установите его командой `%pip install ipywidgets` и перезапустите ядро.

Анимация встроена в вывод и имеет кнопки управления. Вспомогательный код графиков можно свернуть средствами Jupyter.

Под интерактивными графиками меняйте ползунки и нажимайте **«Обновить график»**. Для показа только статических графиков установите `ENABLE_WIDGETS = False` в ячейке настройки.

In [ ]:
%matplotlib inline
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
from scipy.stats import norm, laplace
from scipy.optimize import linprog
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from threadpoolctl import threadpool_limits

ENABLE_WIDGETS = True  # False: только обычные графики
SEED = 42
plt.rcParams.update({
    "figure.figsize": (10, 4), "font.size": 11, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": .2,
    "animation.embed_limit": 30,
})
BLUE, ORANGE, GREEN, RED = "#2563eb", "#ea580c", "#059669", "#dc2626"
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
print("Среда готова. Ползунки:", "доступны" if widgets else "заменены статическими графиками")

In [ ]:
def show_controls(func, **specs):
    """Статический результат сохраняется даже в просмотрщике без виджетов."""
    if widgets is not None and ENABLE_WIDGETS:
        ui = widgets.interactive(func, {"manual": True, "manual_name": "Обновить график"}, **specs)
        display(ui)
    else:
        print("Меняйте аргументы в вызове функции под графиком.")

def mse_value(X, y, w):
    return np.mean((X @ w - y) ** 2)

def gradient_value(X, y, w):
    return 2 / len(y) * X.T @ (X @ w - y)

def run_gd(X, y, w0, lr=.05, steps=100):
    w = np.asarray(w0, dtype=float).copy()
    path, losses = [w.copy()], [mse_value(X, y, w)]
    for _ in range(steps):
        w -= lr * gradient_value(X, y, w)
        loss = mse_value(X, y, w)
        if not np.isfinite(loss) or loss > 1e12:
            break
        path.append(w.copy())
        losses.append(loss)
    return np.asarray(path), np.asarray(losses)

## 1. Как подобрать прямую
### 1.1. Данные с известным ответом
Генератор: $y = 2x + 1 + \varepsilon$. Прямая задаёт систематическую часть, шум — отклонения отдельных наблюдений.

**Вопрос:** если шума станет больше, должны ли истинные коэффициенты измениться? Сможем ли мы восстановить их точно по конечной выборке?

#### Формальная постановка задачи регрессии
Дана обучающая выборка $D=\{(x_i,y_i)\}_{i=1}^n$, где $x_i\in\mathbb R^d$ — признаки, $y_i\in\mathbb R$ — ответ. Для обсуждения обобщения предполагаем, что пары независимо получены из одного распределения $P(X,Y)$; в выводах МНК и правдоподобия можем рассматривать матрицу признаков как фиксированную.

Выбираем функцию из семейства
$$\mathcal F=\{f_{w,b}(x)=w^\top x+b:\ w\in\mathbb R^d,\ b\in\mathbb R\}.$$
Качество на новых объектах описывает **ожидаемый риск**:
$$R(w,b)=\mathbb E_{(X,Y)\sim P}\big[(f_{w,b}(X)-Y)^2\big].$$
Распределение $P$ неизвестно, поэтому минимизируем **эмпирический риск**:
$$\widehat R_D(w,b)=\frac1n\sum_{i=1}^n(f_{w,b}(x_i)-y_i)^2,
\qquad (\hat w,\hat b)\in\arg\min_{w,b}\widehat R_D(w,b).$$

Малое значение $\widehat R_D$ ещё не гарантирует малое $R$: именно поэтому нужны независимые данные для оценки качества.

**Обозначения в ноутбуке:** $d$ — число исходных признаков. После добавления столбца единиц число столбцов матрицы $p=d+1$. В разделах о МНК и GD символ $w$ обозначает весь вектор, включая сдвиг; там, где сдвиг записан отдельно как $b$, $w$ содержит только веса признаков. В коде добавленный столбец единиц и сдвиг всегда последние.

In [ ]:
rng = np.random.default_rng(SEED)
x_line = np.linspace(-2, 2, 60)
noise_line = rng.normal(size=len(x_line))
y_line = 2 * x_line + 1 + .7 * noise_line
X_line = np.column_stack([x_line, np.ones(len(x_line))])  # веса в порядке [a, b]

def plot_noise(sigma=.7):
    fig, ax = plt.subplots()
    ax.scatter(x_line, 2*x_line + 1 + sigma*noise_line, alpha=.65, label="Наблюдения")
    ax.plot(x_line, 2*x_line+1, color=GREEN, lw=2, label="Истина: 2x + 1")
    ax.set(xlabel="x", ylabel="y", title=f"Одна и та же зависимость, шум σ = {sigma:.1f}", ylim=(-8, 10))
    ax.legend(); plt.show()

plot_noise()
show_controls(plot_noise, sigma=(0., 3., .1))

**Что видно на графике.** При изменении $\sigma$ истинная зелёная прямая остаётся прежней, а точки удаляются от неё или приближаются. В коде один набор стандартных случайных ошибок умножается на разные $\sigma$: так сравниваем именно уровень шума, а не одновременно новый шум и новый масштаб. В обучении зелёная прямая была бы неизвестна — доступны только точки.

### 1.2. Обучим модель руками
Модель $\hat y = ax+b$. Подбираем два числа: наклон $a$ и сдвиг $b$.

Остаток: $r_i=\hat y_i-y_i$. Средняя квадратичная ошибка:
$$J(a,b)=\frac1n\sum_{i=1}^n(ax_i+b-y_i)^2 \rightarrow \min_{a,b} $$

**Почему квадрат, а не сумма остатков.** Пусть два остатка равны $-10$ и $10$:
$$r_1+r_2=0,\qquad \frac{r_1^2+r_2^2}{2}=100.$$
Ошибки со знаком компенсировались, хотя каждое предсказание ошибается на 10. Возведение в квадрат исключает такую компенсацию. Если остаток увеличивается с 2 до 10, его вклад в сумму квадратов растёт с 4 до 100 — в 25 раз.

На правом графике каждая точка — **целая модель**, а не отдельный объект. Линии соединяют модели с одинаковой ошибкой. Попробуйте приблизиться к минимуму вручную.

#### От функции потерь к производным
Обозначим $r_i=ax_i+b-y_i$. Тогда по правилу дифференцирования сложной функции:
$$\frac{\partial J}{\partial a}=\frac2n\sum_i r_i x_i,
\qquad \frac{\partial J}{\partial b}=\frac2n\sum_i r_i.$$

При фиксированном $a$ оптимальный сдвиг удовлетворяет
$$\sum_i(ax_i+b-y_i)=0\quad\Rightarrow\quad b=\bar y-a\bar x.$$
Следовательно, прямая МНК со свободным сдвигом проходит через точку $(\bar x,\bar y)$, а сумма её обучающих остатков равна нулю. Это свойство оптимума, а не достаточный критерий хорошей модели.

**Лучшая константа.** Для модели $\hat y=c$:
$$J(c)=\frac1n\sum_i(c-y_i)^2,\qquad
J'(c)=\frac2n\sum_i(c-y_i)=2(c-\bar y).$$
Из $J'(c)=0$ получаем $\hat c=\bar y$. Так как $J''(c)=2>0$, это единственный минимум. Среднее обучающих ответов — естественный базовый прогноз для MSE.

**Эксперимент.** Меняем $a$ и $b$, оставляя наблюдения фиксированными. Вертикальные отрезки слева — остатки, а оранжевая точка справа — выбранные параметры. Зелёная звезда отмечает минимум, вычисленный заранее только как ориентир для демонстрации.

In [ ]:
a_grid = np.linspace(-1, 5, 90)
b_grid = np.linspace(-2, 4, 90)
AA, BB = np.meshgrid(a_grid, b_grid)
JJ = np.mean((AA[..., None]*x_line + BB[..., None] - y_line)**2, axis=-1)
w_best = np.linalg.lstsq(X_line, y_line, rcond=None)[0]

def manual_line(a=.5, b=0.):
    pred = a*x_line+b
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(x_line, y_line, s=18, alpha=.6)
    axes[0].vlines(x_line, y_line, pred, color=ORANGE, alpha=.35)
    axes[0].plot(x_line, pred, color=BLUE, lw=2)
    axes[0].set(xlabel="x", ylabel="y", title=f"Предсказание: MSE = {np.mean((pred-y_line)**2):.3f}", ylim=(-8, 12))
    axes[1].contour(AA, BB, JJ, levels=18, cmap="Blues")
    axes[1].scatter(*w_best, marker="*", s=160, c=GREEN, label="Минимум")
    axes[1].scatter(a, b, s=70, c=ORANGE, label="Текущая модель")
    axes[1].set(xlabel="Наклон a", ylabel="Сдвиг b", title="Пространство параметров")
    axes[1].legend(); plt.tight_layout(); plt.show()

manual_line()
show_controls(manual_line, a=(-1., 5., .1), b=(-2., 4., .1))

**Как читать результат.** Хорошая подгонка слева соответствует попаданию в центр вложенных линий уровня справа. Поворот и перенос прямой — это движение по двум координатам карты ошибки. При этом минимум на шумной выборке может немного отличаться от истинных параметров генератора.

### 1.3. Почему квадраты: вероятностный взгляд
Предположим, что ошибки независимы и имеют одинаковую дисперсию:
$$\varepsilon_i\sim\mathcal N(0,\sigma^2),\qquad
 y_i\mid x_i\sim\mathcal N(ax_i+b,\sigma^2).$$

Модель предсказывает **центр распределения ответа**. Нормальный шум — предположение о данных, а не свойство, автоматически следующее из линейности модели.

На рисунке ниже «колокола» повёрнуты: вертикальная ось — возможное значение $y$, ширина вправо показывает условную плотность. Это иллюстрация, ширина плотности не измеряется в единицах $x$.

In [ ]:
def conditional_gaussians(sigma=.7):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(x_line, y_line, s=16, color="gray", alpha=.35)
    ax.plot(x_line, 2*x_line+1, c=GREEN, label="Условное среднее E[y|x]")
    for x0 in [-1.5, -.5, .5, 1.5]:
        yy = np.linspace(2*x0+1-3*sigma, 2*x0+1+3*sigma, 120)
        density = norm.pdf(yy, loc=2*x0+1, scale=sigma)
        width = .45 * density / density.max()
        ax.fill_betweenx(yy, x0, x0+width, alpha=.3, color=BLUE)
        ax.plot([x0, x0], [yy[0], yy[-1]], c=BLUE, alpha=.4)
    ax.set(xlabel="x", ylabel="Возможный ответ y", title=f"Распределение ответа при фиксированном x; σ = {sigma:.1f}")
    ax.legend(); plt.show()

conditional_gaussians()
show_controls(conditional_gaussians, sigma=(.2, 1.5, .1))

**Смысл рисунка.** Каждая вертикальная линия фиксирует одно значение $x$. Вдоль неё возможны разные ответы, но плотность максимальна около $2x+1$. Ползунок меняет предполагаемый разброс распределений; серые наблюдения в этой демонстрации остаются прежними. Далее будем выбирать наклон и сдвиг по плотности этих наблюдений.

### 1.4. Максимальное правдоподобие → MSE
Теперь у модели есть не только предсказание, но и плотность возможных ответов. Чтобы выбрать параметры, сравним, насколько высокая плотность получается **в наблюдаемых точках**. Параметры, при которых совместная плотность данных максимальна, называются оценкой максимального правдоподобия (MLE).

Правдоподобие $L(a,b)=\prod_i p(y_i\mid x_i,a,b)$ — функция **параметров при фиксированных данных**. Это произведение плотностей, а не вероятность получить в точности данный набор вещественных чисел. Для непрерывной случайной величины вероятность отдельной точки равна нулю.

При фиксированном $\sigma>0$:
$$p(y_i\mid x_i,a,b)=\frac1{\sqrt{2\pi}\sigma}
\exp\left[-\frac{(y_i-ax_i-b)^2}{2\sigma^2}\right].$$

#### От плотности к квадратичной потере
Условно на $X$ пусть $\varepsilon_i$ независимы и $\varepsilon_i\sim\mathcal N(0,\sigma^2)$. Для $\theta=(a,b)$:
$$L(\theta,\sigma^2;D)
=(2\pi\sigma^2)^{-n/2}\exp\left[-\frac1{2\sigma^2}\sum_i(y_i-f_\theta(x_i))^2\right].$$
Применяем логарифм:
$$\ell(\theta,\sigma^2)
=-\frac n2\log(2\pi)-\frac n2\log\sigma^2
-\frac1{2\sigma^2}\sum_i(y_i-f_\theta(x_i))^2.$$
При фиксированном $\sigma^2>0$ первые два слагаемых постоянны по $\theta$, а множитель перед суммой положителен. Отсюда:
$$\arg\max_\theta\ell(\theta,\sigma^2)
=\arg\min_\theta\sum_i(y_i-f_\theta(x_i))^2
=\arg\min_\theta J(\theta).$$

**Итог вывода:** гауссовский шум с общей фиксированной дисперсией приводит к тому же решению, что MSE. Деление суммы квадратов на $n$ не меняет минимум, но меняет масштаб градиента.

**Пример: какую прямую предпочитают три наблюдения?**

| Объект | Признак $x_i$ | Наблюдаемый ответ $y_i$ |
|---|---:|---:|
| 1 | −1 | −0.8 |
| 2 | 0 | 1.4 |
| 3 | 1 | 2.7 |

Рассматриваем прямые $\hat y=ax+1$: меняем **только наклон $a$**, а сдвиг $b=1$ и шум $\sigma=0.7$ фиксируем. При $a=0.5$ предсказания равны $(0.5,1,1.5)$; при $a=2$ — $(-1,1,3)$. Во втором случае они ближе к фактическим ответам.

**Первая фигура: насколько наблюдаемый ответ согласуется с предсказанием?** Для каждого объекта выделен свой график. На горизонтальной оси теперь **ответ $y$, а не признак $x$**. Значение $x_i$ фиксировано и указано в заголовке.

- Синий колокол — предполагаемое распределение ответа вокруг предсказания $\hat y_i=ax_i+1$; синяя пунктирная линия проходит через его центр.
- Красная вертикальная линия — фактический ответ $y_i$, который при изменении модели остаётся на месте.
- Красная точка на колоколе показывает **плотность** в наблюдаемом ответе. Чем ближе ответ к центру, тем выше эта точка. Высота — плотность, не вероятность отдельного значения.

Перемещая ползунок $a$, мы двигаем колокола относительно фиксированных красных линий. У объекта с $x=0$ предсказание всегда равно 1, поэтому его график не меняется.

**Вторая фигура: как выбрать один наклон для всех объектов?** Здесь на горизонтальной оси уже **параметр $a$**. Слева считаем MSE для каждого наклона, справа — $-\log L$, то есть сумму отрицательных логарифмов трёх плотностей. У каждой кривой своя панель и одна вертикальная ось. Оба критерия нужно минимизировать. Зелёная линия отмечает лучший наклон, оранжевая точка — выбранный ползунком.


In [ ]:
x_small = np.array([-1., 0., 1.])
y_small = np.array([-.8, 1.4, 2.7])

def likelihood_demo(a=.5):
    b, sigma = 1., .7  # Меняем только наклон, чтобы сравнение оставалось однозначным.
    predictions = a*x_small+b
    densities = norm.pdf(y_small, predictions, sigma)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
    yy = np.linspace(-6, 7, 700)
    for k, (ax, xi, yi, mu, density) in enumerate(zip(axes, x_small, y_small, predictions, densities), 1):
        ax.plot(yy, norm.pdf(yy, mu, sigma), color=BLUE, lw=2)
        ax.axvline(mu, color=BLUE, ls="--", label=f"Предсказание ŷ = {mu:.2f}")
        ax.vlines(yi, 0, density, color=RED, lw=2, label=f"Фактический y = {yi:g}")
        ax.scatter(yi, density, color=RED, s=55, zorder=5)
        ax.set(xlabel="Возможный ответ y (не признак x)",
               title=f"Объект {k}: x = {xi:g}\nПлотность в ответе: {density:.3f}",
               xlim=(-6,7), ylim=(0,.68))
        ax.legend(loc="upper right", fontsize=9)
    axes[0].set_ylabel("Плотность p(y | x, a)")
    fig.suptitle(f"1. Три объекта по отдельности: наклон a = {a:.2f}, b = 1, σ = 0.7", fontsize=13)
    fig.tight_layout(); plt.show()

    slopes = np.linspace(-1, 5, 301)
    all_predictions = slopes[:, None]*x_small+b
    errors = np.mean((all_predictions-y_small)**2, axis=1)
    nlls = -norm.logpdf(y_small, all_predictions, sigma).sum(axis=1)
    # Точный оптимум по a при фиксированном b, без округления к узлам сетки.
    best_a = np.dot(x_small, y_small-b)/np.dot(x_small, x_small)
    current_mse = np.mean((predictions-y_small)**2)
    current_nll = -norm.logpdf(y_small, predictions, sigma).sum()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
    for ax, values, current, color, title, ylabel in [
        (axes[0], errors, current_mse, BLUE, "Суммируем квадраты ошибок", "MSE — меньше лучше"),
        (axes[1], nlls, current_nll, RED, "Суммируем −log плотностей", "−log L — меньше лучше"),
    ]:
        ax.plot(slopes, values, color=color, lw=2)
        ax.axvline(best_a, color=GREEN, ls="--", label=f"Минимум: a = {best_a:.2f}")
        ax.scatter(a, current, color=ORANGE, s=70, zorder=5, label=f"Выбрано: a = {a:.2f}")
        ax.set(xlabel="Наклон прямой a (не ответ y)", ylabel=ylabel, title=title, xlim=(-1,5))
        ax.legend(fontsize=9)
    fig.suptitle("2. Все объекты вместе: два критерия выбирают один наклон", fontsize=13)
    fig.tight_layout(); plt.show()
    print(f"Выбрано a={a:.2f}: MSE={current_mse:.3f}; −log L={current_nll:.3f}.")
    print(f"Оба критерия минимальны при a={best_a:.2f} (при фиксированном b=1).")

likelihood_demo()
show_controls(likelihood_demo, a=(-1., 5., .05))

**Попробуйте $a=0.5$, затем $a=1.75$ и $a=4$.** В первом и третьем случаях некоторые наблюдаемые ответы находятся далеко от центров своих колоколов: плотности малы, а ошибки велики. При $a=1.75$ достигается лучший общий компромисс для этих трёх объектов при $b=1$.

На второй фигуре зелёные линии стоят на одном значении $a$, хотя числа на вертикальных осях разные. Это и есть проверяемое утверждение: **MSE и гауссовское правдоподобие выбирают одинаковые коэффициенты**, а не имеют одинаковые численные значения. Здесь проверяем только наклон при фиксированном сдвиге; выведенная выше формула работает и при совместном выборе наклона и сдвига.

Поскольку здесь $n=3$ и $\sigma=0.7$, связь критериев можно записать численно:
$$-\log L=3\log(\sqrt{2\pi}\cdot0.7)+\frac3{2\cdot0.7^2}\operatorname{MSE}
\approx1.687+3.061\operatorname{MSE}.$$
Прибавление константы и умножение на положительное число меняют высоту кривой, но не положение её минимума.

### 1.5. ★ Другая модель шума — другая потеря
Для шума Лапласа $p(\varepsilon)\propto\exp(-|\varepsilon|/s)$ отрицательное логарифмическое правдоподобие даёт сумму **абсолютных** ошибок (MAE после нормировки).

Добавим выброс по ответу $y$ к обычному объекту. Квадратичная потеря сильнее реагирует на большой остаток. Устойчивость MAE к таким выбросам не означает устойчивость ко всем видам выбросов, например к экстремальным значениям признаков.

#### Формализация MAE через распределение Лапласа
Для независимых ошибок с плотностью $p(\varepsilon)=\frac1{2s}\exp(-|\varepsilon|/s)$ при фиксированном $s>0$:
$$-\ell(\theta)=n\log(2s)+\frac1s\sum_i|y_i-f_\theta(x_i)|.$$
Поэтому MLE эквивалентен минимизации MAE. Для постоянного предсказания минимум MAE достигается на медиане выборки; при чётном числе объектов минимум может быть не единственным. На уровне условного распределения MAE ориентируется на условную медиану, MSE — на условное среднее.

**Эксперимент с выбросом.** Добавляем один объект с фиксированным x = 1.5 и управляем только его ответом. Синяя прямая минимизирует MSE, оранжевая — MAE. Для MAE используется готовый решатель линейной оптимизации; здесь рассматриваем результат его работы. Правый график показывает, почему эти две модели по-разному реагируют на один большой остаток.

In [ ]:
def fit_mae_line(X, y):
    # min sum(t_i), ограничения -t <= Xw-y <= t. Это точное LP, не сглаживание MAE.
    n, p = X.shape
    result = linprog(np.r_[np.zeros(p), np.ones(n)],
                     A_ub=np.vstack([np.c_[X, -np.eye(n)], np.c_[-X, -np.eye(n)]]),
                     b_ub=np.r_[y, -y], bounds=[(None, None)]*p+[(0, None)]*n,
                     method="highs")
    if not result.success:
        raise RuntimeError(result.message)
    return result.x[:p]

def outlier_demo(height=12.):
    xx = np.r_[x_line, 1.5]
    yy = np.r_[y_line, height]
    X = np.c_[xx, np.ones(len(xx))]
    w_mse = np.linalg.lstsq(X, yy, rcond=None)[0]
    w_mae = fit_mae_line(X, yy)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(xx[:-1], yy[:-1], alpha=.4, s=15)
    axes[0].scatter(xx[-1], yy[-1], c=RED, s=70, label="Меняем один ответ")
    for w, color, name in [(w_mse, BLUE, "MSE"), (w_mae, ORANGE, "MAE")]:
        axes[0].plot(x_line, X_line@w, c=color, label=name)
    axes[0].legend(); axes[0].set(xlabel="x", ylabel="y", title="Реакция на выброс")
    r = np.linspace(-4, 4, 200)
    axes[1].plot(r, r*r, label="Квадрат остатка")
    axes[1].plot(r, abs(r), label="Модуль остатка")
    axes[1].set(xlabel="Остаток", ylabel="Потеря", title="Большие ошибки имеют разный вес")
    axes[1].legend(); plt.tight_layout(); plt.show()

outlier_demo()
show_controls(outlier_demo, height=(4., 35., 1.))

**Что меняется.** Поднимая красную точку, заметнее сдвигаем решение MSE: большой остаток имеет большой вес в квадратичном критерии. MAE реагирует слабее. Это иллюстрация влияния выброса, а не доказательство, что MAE всегда даёт лучший прогноз: выбор потери зависит от задачи и характера ошибок.

## 2. Можно ли найти решение сразу
### 2.1. Матричная запись
Пусть $X\in\mathbb R^{n\times p}$, $w\in\mathbb R^p$, $y\in\mathbb R^n$. В $X$ при необходимости уже включён столбец единиц для свободного коэффициента.
$$\hat y=Xw,\qquad J(w)=\frac1n\|Xw-y\|_2^2.$$

**Проверка понимания:** какие размерности имеют $Xw$, $X^\top y$ и $X^\top X$?


### 2.2. Нормальные уравнения
Переходим от перебора прямых к вычислению оптимальных весов. Для этого найдём точку, в которой все частные производные ошибки равны нулю. Сначала выведем градиент, затем проверим, почему его нуль действительно соответствует минимуму.

#### Градиент, минимум и единственность решения
Раскроем квадрат, учитывая, что результат — скаляр:
$$J(w)=\frac1n(w^\top X^\top Xw-2w^\top X^\top y+y^\top y).$$
Матрица $X^\top X$ симметрична. Используем $\nabla_w(w^\top Aw)=2Aw$ для симметричной $A$ и $\nabla_w(w^\top c)=c$:
$$\nabla J(w)=\frac2nX^\top Xw-\frac2nX^\top y,
\qquad H=\nabla^2J(w)=\frac2nX^\top X.$$
Для любого вектора $u$:
$$u^\top Hu=\frac2n\|Xu\|^2\ge0.$$
Значит, $J$ выпукла и любая стационарная точка — глобальный минимум. Строгая выпуклость и единственность весов имеют место тогда и только тогда, когда $\operatorname{rank}(X)=p$.

В минимуме градиент равен нулю:
$$\frac2nX^\top(Xw-y)=0
\quad\Longleftrightarrow\quad X^\top Xw=X^\top y.$$
При полном ранге столбцов матрица $X^\top X$ обратима:
$$\hat w=(X^\top X)^{-1}X^\top y.$$

### 2.3. Формула ≠ инструкция обязательно обращать матрицу
`inv` — буквальная формула; `solve` — решение системы без явного вычисления обратной; `lstsq` — прямое решение задачи наименьших квадратов, в том числе при недостаточном ранге.

Формирование $X^\top X$ ухудшает обусловленность: при полном ранге $\kappa_2(X^\top X)=\kappa_2(X)^2$. Поэтому в реальном коде для МНК используем `lstsq`, а не нормальные уравнения.

**Что проверяем.** Сначала вычисляем веса тремя способами на хорошо обусловленной задаче и сравниваем MSE. Затем добавляем точную копию признака: число столбцов растёт, но ранг — нет. Во второй части используем `lstsq`, поскольку формула с обратной матрицей уже неприменима.

In [ ]:
XtX, Xty = X_line.T @ X_line, X_line.T @ y_line
solutions = {
    "inv (учебная формула)": np.linalg.inv(XtX) @ Xty,
    "solve (нормальные уравнения)": np.linalg.solve(XtX, Xty),
    "lstsq (предпочтительный вариант)": np.linalg.lstsq(X_line, y_line, rcond=None)[0],
}
for name, w in solutions.items():
    print(f"{name:35s}: a={w[0]:.4f}, b={w[1]:.4f}, MSE={mse_value(X_line,y_line,w):.4f}")

X_duplicate = np.c_[x_line, x_line, np.ones(len(x_line))]
print("\nРанг с дублирующим столбцом:", np.linalg.matrix_rank(X_duplicate), "из", X_duplicate.shape[1])
w_duplicate = np.linalg.lstsq(X_duplicate, y_line, rcond=None)[0]
print("lstsq всё равно даёт решение:", w_duplicate.round(4))
print("Два коэффициента при x определяются только через их сумму.")

**Вывод из вычислений.** В первом примере методы дают одинаковые коэффициенты с точностью вычислений. После дублирования столбца нельзя однозначно разделить его эффект между двумя весами: в предсказание входит их сумма. Отсутствие обратной матрицы не означает, что невозможно подобрать предсказания методом наименьших квадратов.

### 2.4. Цена прямого решения
Для плотной матрицы: формирование $X^\top X$ требует порядка $np^2$ операций, решение системы — порядка $p^3$, один шаг полного GD — порядка $np$.

Ниже — **замер на вашем компьютере**, а не заранее придуманные времена. Фиксируем число объектов, увеличиваем число признаков. Ограничиваем BLAS одним потоком для более стабильного сравнения. Генерация данных не входит в замеры, показываем медиану трёх повторов.

**Важно:** один шаг GD не равен обучению. Для $k$ шагов стоимость порядка $knp$; число шагов зависит от геометрии задачи и точности. Для небольших задач прямое решение часто предпочтительнее.

При желании включите `HEAVY_BENCHMARK = True`: этот запуск может занять заметное время.

In [ ]:
HEAVY_BENCHMARK = False
sizes = [32, 64, 128, 256, 512] if not HEAVY_BENCHMARK else [128, 256, 512, 1024, 1536]
n_bench = 1800 if not HEAVY_BENCHMARK else 2400

def median_time(fn, repeats=3):
    fn()  # прогрев
    measurements = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        measurements.append(time.perf_counter()-start)
    return np.median(measurements)

benchmark = []
with threadpool_limits(limits=1):
    for p in sizes:
        local_rng = np.random.default_rng(SEED+p)
        X = local_rng.normal(size=(n_bench, p))
        y = local_rng.normal(size=n_bench)
        gram, rhs = X.T@X, X.T@y
        w = np.zeros(p)
        row = [p,
               median_time(lambda: X.T@X),
               median_time(lambda: np.linalg.inv(gram)@rhs),
               median_time(lambda: np.linalg.solve(gram, rhs)),
               median_time(lambda: gradient_value(X, y, w)),
               median_time(lambda: np.linalg.lstsq(X, y, rcond=None))]
        benchmark.append(row)
benchmark = np.asarray(benchmark)
fig, ax = plt.subplots(figsize=(10, 4))
for j, label in enumerate(["Формирование XᵀX", "inv готовой XᵀX", "solve готовой системы", "Один градиент", "lstsq: всё решение"], 1):
    ax.semilogy(benchmark[:, 0], 1000*benchmark[:, j], "o-", label=label)
ax.set(xlabel="Число признаков p", ylabel="Медианное время, мс (логарифмическая шкала)", title=f"Цена операций: n={n_bench}, один поток BLAS")
ax.legend(fontsize=9); plt.show()
print(" p | XᵀX, мс | inv, мс | solve, мс | 1 градиент, мс | lstsq, мс")
for row in benchmark:
    print(f"{int(row[0]):4d} | " + " | ".join(f"{1000*v:9.3f}" for v in row[1:]))

**Как сравнивать времена.** Линии `inv` и `solve` начинаются с уже готовой матрицы, поэтому для полной стоимости нормальных уравнений нужно добавить её формирование и вычисление правой части. `lstsq` измеряет сразу решение всей задачи. «Один градиент» — только одна операция будущего итеративного алгоритма; её дешевизна ещё не доказывает, что всё обучение окажется быстрее. Замер мотивирует попробовать последовательные небольшие обновления вместо одного прямого решения.

## 3. Градиентный спуск
Мы уже умеем находить минимум MSE решением линейной задачи. Но замеры показали, что стоимость прямого решения растёт с числом признаков. Кроме того, для многих других функций потерь такой удобной формулы решения нет.

Возникает другая стратегия: **взять любое начальное предсказание и постепенно исправлять его параметры**. В разделе 1 мы делали это руками, двигая прямую. Теперь нужна формальная инструкция: в какую сторону менять каждый коэффициент и насколько сильно.

Градиентный спуск реализует эту стратегию. Он не меняет модель и функцию потерь: мы по-прежнему ищем минимум той же MSE. Меняется только способ поиска. На нашем небольшом примере решение `lstsq` будет эталоном для проверки.

### 3.1. Как выбрать направление изменения весов
Производная показывает, как меняется ошибка при малом увеличении параметра. Если производная положительна, небольшое уменьшение параметра уменьшает ошибку; если отрицательна — нужно увеличить параметр. Для нескольких параметров частные производные собираются в вектор — градиент.

**От одной прямой к обновлению параметров.** Для $r_i=ax_i+b-y_i$:
$$J(a,b)=\frac1n\sum_i r_i^2,$$
$$\frac{\partial J}{\partial a}=\frac1n\sum_i2r_i\frac{\partial r_i}{\partial a}
=\frac2n\sum_i r_ix_i,\qquad
\frac{\partial J}{\partial b}=\frac2n\sum_i r_i.$$
Обе производные вычисляются в одной текущей точке $(a_t,b_t)$, после чего параметры обновляются одновременно:
$$a_{t+1}=a_t-\eta\frac2n\sum_i(a_tx_i+b_t-y_i)x_i,$$
$$b_{t+1}=b_t-\eta\frac2n\sum_i(a_tx_i+b_t-y_i).$$
Объединяя $a,b$ в вектор весов и добавляя столбец единиц, получаем:
$$w_{t+1}=w_t-\eta\nabla J(w_t),\qquad
\nabla J(w)=\frac2nX^\top(Xw-y).$$

Положительное число $\eta$ — **шаг обучения**. Направление задаёт градиент, а шаг определяет масштаб перемещения. Слишком большой шаг может перенести нас через область минимума и увеличить ошибку. Ниже обоснуем направление спуска, затем посмотрим реализацию и сам процесс обучения.

#### Почему отрицательный градиент — направление спуска
Для малого приращения $h$:
$$J(w+h)=J(w)+\nabla J(w)^\top h+o(\|h\|).$$
Среди направлений $d$ с $\|d\|_2=1$ скалярное произведение $\nabla J(w)^\top d$ минимально при
$$d=-\frac{\nabla J(w)}{\|\nabla J(w)\|_2},\qquad \nabla J(w)\ne0.$$
Это следует из неравенства Коши — Буняковского. Выбор $h=-\eta\nabla J(w)$ даёт изменение первого порядка $-\eta\|\nabla J(w)\|^2$.

**Ограничение аргумента:** линейное приближение локально. При большом $\eta$ члены более высокого порядка могут перевесить уменьшение, и ошибка возрастёт.

Для квадратичной функции остаточный член известен точно:
$$J(w-\eta g)-J(w)=-\eta\|g\|^2+\frac{\eta^2}2g^\top Hg,
\qquad g=\nabla J(w).$$
Это связывает безопасный шаг с кривизной поверхности.

### 3.2. Весь алгоритм в нескольких строках
Начинаем с заведомо неточной прямой: $a=-0.8$, $b=-1.5$. Далее 80 раз повторяем четыре действия:

1. `X_line @ w` — предсказания текущей модели для всех объектов.
2. `prediction - y_line` — вектор остатков.
3. `X_line.T @ (...)` с множителем $2/n$ — градиент MSE по двум параметрам.
4. `w - learning_rate * gradient` — новые веса.

`history_w` и `history_loss` сохраняют начальное состояние и результат каждого шага. Они нужны для следующей анимации, а не для вычисления обновлений. Решение `lstsq` внутри цикла не используется.

In [ ]:
w = np.array([-.8, -1.5])
learning_rate = .1
history_w = [w.copy()]
history_loss = [mse_value(X_line, y_line, w)]
for step in range(80):
    prediction = X_line @ w
    gradient = 2 / len(y_line) * X_line.T @ (prediction - y_line)
    w = w - learning_rate * gradient
    history_w.append(w.copy())
    history_loss.append(mse_value(X_line, y_line, w))
history_w = np.asarray(history_w)
history_loss = np.asarray(history_loss)
print("GD:   ", w.round(5))
print("lstsq:", w_best.round(5))

**Результат первого запуска.** GD получил практически те же веса, что `lstsq`, используя только предсказания, остатки и градиент. Так мы проверяем корректность алгоритма на задаче, где ответ уже известен. Истинные параметры генератора и оптимальные веса на конечной шумной выборке могут различаться.

### 3.3. Один процесс — три представления
Выведенные выше коэффициенты GD близки к решению `lstsq`. Теперь посмотрим, **как** алгоритм к ним пришёл, используя уже сохранённую историю — повторного обучения в этой ячейке нет.

Слева перемещается прямая среди объектов. В центре та же модель показана точкой $(a,b)$ на карте ошибки; оранжевый след соединяет последовательные состояния. Справа накапливаются значения MSE. Зелёные ориентиры соответствуют решению МНК.

В начале параметры быстро меняются. Вблизи минимума градиент становится малым, поэтому обновления сокращаются. Ошибка не обязана дойти до нуля: в данных есть шум, который одна прямая не может воспроизвести. Кнопки ▶ и пауза позволяют рассмотреть первые шаги отдельно.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
axes[0].scatter(x_line, y_line, s=12, alpha=.5)
axes[0].plot(x_line, X_line@w_best, c=GREEN, ls="--", label="Решение МНК")
line, = axes[0].plot([], [], c=BLUE, lw=2)
axes[0].set(xlim=(-2.1, 2.1), ylim=(-5, 7), xlabel="x", ylabel="y", title="Прямая учится")
axes[0].legend(fontsize=8)
axes[1].contour(AA, BB, JJ, levels=20, cmap="Blues")
axes[1].scatter(*w_best, c=GREEN, marker="*", s=120)
trail, = axes[1].plot([], [], "o-", c=ORANGE, markersize=3)
axes[1].set(xlabel="a", ylabel="b", title="Траектория параметров")
curve, = axes[2].plot([], [], c=BLUE)
axes[2].axhline(mse_value(X_line, y_line, w_best), c=GREEN, ls="--")
axes[2].set(xlim=(0, 80), ylim=(.1, history_loss.max()*1.2), yscale="log", xlabel="Шаг", ylabel="MSE", title="Ошибка обучения")
label = fig.suptitle("")
fig.tight_layout()

def animate_step(i):
    line.set_data(x_line, X_line@history_w[i])
    trail.set_data(history_w[:i+1, 0], history_w[:i+1, 1])
    curve.set_data(np.arange(i+1), history_loss[:i+1])
    label.set_text(f"Шаг {i}: a={history_w[i,0]:.3f}, b={history_w[i,1]:.3f}")
    return line, trail, curve, label

animation = FuncAnimation(fig, animate_step, frames=range(0, 81, 2), interval=140, blit=False)
animation_html = animation.to_jshtml()
plt.close(fig)
display(HTML(animation_html))

### 3.4. Маленький, подходящий и слишком большой шаг
Запускаем алгоритм из одной точки. Сравниваем не только конечный ответ, но и весь путь.

**Как изменится обучение:** какое поведение будет при $\eta=0.005$, $0.1$ и $0.9$?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].contour(AA, BB, JJ, levels=20, cmap="Greys", alpha=.5)
for lr, color in [(.005, BLUE), (.1, GREEN), (.9, RED)]:
    path, losses = run_gd(X_line, y_line, [-.8, -1.5], lr=lr, steps=70)
    axes[0].plot(path[:, 0], path[:, 1], ".-", c=color, label=f"η={lr}", markersize=3)
    axes[1].semilogy(losses, c=color, label=f"η={lr}")
axes[0].set(xlim=(-1, 5), ylim=(-2, 4), xlabel="a", ylabel="b", title="Расходящийся путь выходит за границы")
axes[1].set(xlabel="Шаг", ylabel="MSE (логарифмическая шкала)", title="Цена неверного шага")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

H = 2/len(y_line) * X_line.T@X_line
eta_limit = 2/np.linalg.eigvalsh(H).max()
print(f"Для этой квадратичной задачи диапазон гарантированной сходимости: 0 < η < {eta_limit:.3f}")

**Как читать три режима.** При η = 0.005 ошибка убывает медленно. При η = 0.1 быстро приближается к минимальному уровню. При η = 0.9 обновления перелетают через минимум и ошибка растёт; на карте часть траектории выходит за границы. Логарифмическая ось ошибки позволяет показать все режимы вместе. Число под графиком — вычисленная для этой матрицы верхняя граница безопасного постоянного шага, а не универсальное значение для других данных.

### 3.5. Когда остановиться
- Бюджет итераций — ограничение времени, но не гарантия точности.
- Малая норма градиента — локальный признак близости к стационарной точке.
- Малое изменение ошибки — полезный сигнал, но при слишком маленьком шаге оно бывает и далеко от минимума.

В этом примере можно сравнить с `lstsq`, потому что точное решение доступно. В больших задачах обычно заранее задают бюджет и критерий остановки.

## 4. Почему скорость обучения зависит от данных
### 4.1. Один признак измеряется в метрах, другой — в сантиметрах
Меняем масштаб одного столбца. Задача остаётся линейной, но поверхность ошибки вытягивается. Шаг, безопасный для крутого направления, даёт медленное движение вдоль пологого.

### 4.2. Стандартизация меняет геометрию
В каждой системе координат используем безопасный шаг, вычисленный по кривизне. Сравниваем относительный избыток ошибки над минимумом. Это отделяет проблему обусловленности от случайного выбора шага.

**Что делает код.** Берём два исходно сопоставимых признака и умножаем второй на 20. Для обеих версий вычисляем решение МНК как ориентир и запускаем GD из нулевых весов. Шаг в каждом случае выбирается по гессиану автоматически, чтобы оба запуска были устойчивыми.

На первых двух панелях коэффициенты выражены в разных единицах, поэтому их числа напрямую не сравниваем. Справа рисуем $(J_t-J^*)/(J_0-J^*)$: долю начального избытка ошибки, которая ещё осталась. В заголовках κ(H) характеризует различие кривизны в разных направлениях: большое значение соответствует вытянутой поверхности.

In [ ]:
rng_scale = np.random.default_rng(SEED)
base = rng_scale.normal(size=(300, 2))
base -= base.mean(axis=0)
X_unscaled = base * [1., 20.]
y_scale = X_unscaled @ np.array([2., .1]) + rng_scale.normal(0, .3, 300)
scaler_demo = StandardScaler().fit(X_unscaled)
X_scaled = scaler_demo.transform(X_unscaled)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, (X, title, color) in enumerate([(X_unscaled, "Исходные признаки", ORANGE), (X_scaled, "После стандартизации", BLUE)]):
    optimum = np.linalg.lstsq(X, y_scale, rcond=None)[0]
    H = 2/len(X)*X.T@X
    safe_lr = .9/np.linalg.eigvalsh(H).max()
    path, losses = run_gd(X, y_scale, np.zeros(2), safe_lr, 180)
    u = np.linspace(-.3, max(2.5, optimum[0]*1.3), 100)
    v = np.linspace(-.08, max(.22, optimum[1]*1.3), 100)
    U, V = np.meshgrid(u, v)
    points = np.stack([U, V], axis=-1)
    delta = points-optimum
    J = .5*np.einsum('...i,ij,...j->...', delta, H, delta)
    axes[idx].contour(U, V, J, levels=np.geomspace(max(J.max()*1e-4, 1e-5), J.max(), 15), cmap="Blues")
    axes[idx].plot(path[:,0], path[:,1], ".-", c=color, markersize=2)
    axes[idx].scatter(*optimum, marker="*", s=100, c=GREEN)
    axes[idx].set(xlabel="w₁ в своей системе координат", ylabel="w₂", title=f"{title}\nκ(H)={np.linalg.cond(H):.1f}")
    best_loss = mse_value(X,y_scale,optimum)
    gap = np.maximum((losses-best_loss)/(losses[0]-best_loss), 1e-15)
    axes[2].semilogy(gap, c=color, label=title)
axes[2].set(xlabel="Шаг", ylabel="(J − J*) / (J₀ − J*)", title="Относительный избыток ошибки")
axes[2].legend(fontsize=8); plt.tight_layout(); plt.show()

**Результат стандартизации.** После выравнивания масштабов GD быстрее уменьшает избыток ошибки. В исходных координатах шаг ограничен крутым направлением, а вдоль пологого направления продвижение остаётся медленным. Нижняя горизонтальная часть синей кривой — установленный в коде порог отображения 10⁻¹⁵, а не измерение точной ошибки ниже машинной точности.

### 4.3. Полный градиент или случайный батч
Для батча $B$:
$$g_B(w)=\frac2{|B|}X_B^\top(X_Bw-y_B).$$
Случайный батч даёт шумную оценку полного градиента. Маленький батч дешевле на шаг, но траектория менее гладкая. При постоянном шаге SGD может колебаться около минимума.

Слева сравниваем шаги обновления, справа — суммарное число обработанных обучающих объектов. Правая ось учитывает размер батча, но **не является временем исполнения**: векторизация и оборудование тоже важны. Расчёт полной ошибки нужен только для визуализации и в счётчик не включён.

**Что меняем в эксперименте.** Сохраняем начальные веса и шаг, но вычисляем обновления по одному объекту, десяти объектам или всей выборке. Для каждого шага случайно выбираем новый батч. После обновления считаем MSE на всей выборке, чтобы кривые качества были сопоставимы.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for batch_size, color in [(1, ORANGE), (10, BLUE), (len(y_line), GREEN)]:
    batch_rng = np.random.default_rng(SEED)
    w = np.array([-.8, -1.5])
    losses = [mse_value(X_line, y_line, w)]
    for step in range(250):
        ids = batch_rng.choice(len(y_line), size=batch_size, replace=False)
        w -= .05 * gradient_value(X_line[ids], y_line[ids], w)
        losses.append(mse_value(X_line, y_line, w))
    axes[0].semilogy(losses, c=color, label=f"Батч {batch_size}")
    axes[1].semilogy(np.arange(len(losses))*batch_size/len(y_line), losses, c=color, label=f"Батч {batch_size}")
for ax in axes:
    ax.axhline(mse_value(X_line,y_line,w_best), c="gray", ls=":")
    ax.set_ylabel("MSE на всей обучающей выборке"); ax.legend()
axes[0].set(xlabel="Обновления весов", title="Стоимость обновлений разная")
axes[1].set(xlabel="Обработанные объекты / n", title="Эквивалентные проходы по данным", xlim=(0, 25))
plt.tight_layout(); plt.show()

**Что видно.** Полный градиент даёт гладкое уменьшение ошибки, небольшие батчи — колебания. По оси обработанных объектов малые батчи здесь быстрее достигают области хороших решений, но продолжают шуметь. Это результат конкретного эксперимента, а не гарантия преимущества маленького батча на любой задаче.

## 5. Регуляризация
### 5.1. Почти одинаковые признаки — нестабильные веса
Пусть $x_2\approx x_1$. Модели с весами $(2,0)$, $(12,-10)$ и $(-8,10)$ могут давать похожие предсказания: важна преимущественно сумма весов.

В эксперименте фиксируем признаки и многократно слегка меняем ответы. Рассматриваем распределение оценок коэффициентов. Для наглядности признаки центрированы, свободный коэффициент не нужен.

**Что сравниваем.** Матрица признаков одинакова во всех 80 повторах; меняется только шум в ответах. Для каждого повтора обучаем обычную регрессию и Ridge. Слева каждая точка — пара обученных весов, справа сравниваем отдельный вес и сумму весов. Ridge здесь впервые показан как способ стабилизировать решение; его критерий выведем в разделе 5.3.

In [ ]:
rng_col = np.random.default_rng(SEED)
z = rng_col.normal(size=100)
X_col = np.c_[z, z + .015*rng_col.normal(size=100)]
X_col -= X_col.mean(axis=0)
y_col_true = 2*X_col[:,0]
ols_weights, ridge_weights = [], []
for _ in range(80):
    y = y_col_true + rng_col.normal(0, .3, len(z))
    ols_weights.append(np.linalg.lstsq(X_col,y,rcond=None)[0])
    ridge_weights.append(np.linalg.solve(X_col.T@X_col + len(z)*.1*np.eye(2), X_col.T@y))
ols_weights, ridge_weights = np.array(ols_weights), np.array(ridge_weights)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ols_weights[:,0], ols_weights[:,1], alpha=.5, label="МНК", c=ORANGE)
axes[0].scatter(ridge_weights[:,0], ridge_weights[:,1], alpha=.6, label="Ridge", c=BLUE)
axes[0].set(xlabel="w₁", ylabel="w₂", title="80 слегка разных обучающих выборок")
axes[0].legend()
axes[1].boxplot([ols_weights[:,0], ols_weights.sum(axis=1), ridge_weights[:,0], ridge_weights.sum(axis=1)],
                positions=[1, 2, 3, 4])
axes[1].set_xticks([1, 2, 3, 4], ["МНК: w₁", "МНК: w₁+w₂", "Ridge: w₁", "Ridge: w₁+w₂"])
axes[1].axhline(2, c=GREEN, ls=":", label="Истинный суммарный эффект")
axes[1].set(ylabel="Значение", title="Отдельный вес и суммарный эффект")
axes[1].tick_params(axis="x", labelsize=9)
plt.tight_layout(); plt.show()

**Интерпретация.** У МНК отдельные коэффициенты сильно меняются и компенсируют друг друга, тогда как их сумма стабильнее. Ridge предпочитает небольшие коэффициенты, поэтому его решения группируются теснее. Меньшая чувствительность к шуму покупается изменением самой задачи оптимизации.

### 5.2. Как линейная модель переобучается
«Линейная» означает линейность **по весам**. Модель $w_0+w_1x+w_2x^2+\dots+w_dx^d$ линейна по параметрам, хотя рисует кривую.

Обучаем полиномы на малой шумной выборке. Валидационные наблюдения независимы, но получены из той же зависимости. На реальных данных истинная кривая неизвестна.

**Устройство эксперимента.** Генерируем 22 обучающих и 250 валидационных объектов из одной функции sin(3x) с независимым шумом. Слева сравниваем полиномы степеней 1, 3 и 15, справа перебираем степени от 1 до 15. `PolynomialFeatures` создаёт степени x, а `LinearRegression` подбирает их веса. Обработка признаков обучается только на обучающей выборке.

In [ ]:
rng_poly = np.random.default_rng(SEED)
x_train_poly = np.sort(rng_poly.uniform(-1, 1, 22)).reshape(-1,1)
x_valid_poly = rng_poly.uniform(-1, 1, 250).reshape(-1,1)
def true_curve(x): return np.sin(3*x).ravel()
y_train_poly = true_curve(x_train_poly)+rng_poly.normal(0,.25,len(x_train_poly))
y_valid_poly = true_curve(x_valid_poly)+rng_poly.normal(0,.25,len(x_valid_poly))
x_plot_poly = np.linspace(-1,1,400).reshape(-1,1)

def polynomial_model(degree, alpha=0.):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(),
                         LinearRegression() if alpha == 0 else Ridge(alpha=alpha))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for degree, color in [(1, ORANGE), (3, GREEN), (15, BLUE)]:
    model = polynomial_model(degree).fit(x_train_poly, y_train_poly)
    axes[0].plot(x_plot_poly, model.predict(x_plot_poly), c=color, label=f"Степень {degree}")
axes[0].scatter(x_train_poly, y_train_poly, c="black", s=22)
axes[0].plot(x_plot_poly, true_curve(x_plot_poly), c="gray", ls="--", label="Истина")
axes[0].set(xlabel="x", ylabel="y", ylim=(-2,2), title="Сложная кривая может подгонять шум")
axes[0].legend(fontsize=9)
train_errors, valid_errors = [], []
for degree in range(1,16):
    model = polynomial_model(degree).fit(x_train_poly, y_train_poly)
    train_errors.append(np.mean((model.predict(x_train_poly)-y_train_poly)**2))
    valid_errors.append(np.mean((model.predict(x_valid_poly)-y_valid_poly)**2))
axes[1].semilogy(range(1,16), train_errors, "o-", label="Обучение")
axes[1].semilogy(range(1,16), valid_errors, "o-", label="Валидация")
axes[1].set(xlabel="Степень полинома", ylabel="MSE", title="Низкая ошибка обучения не гарантирует обобщение")
axes[1].legend(); plt.tight_layout(); plt.show()

**Как читать графики.** Прямая слишком проста для синусоидальной зависимости. Полином высокой степени может подогнать шум и резко отклоняться между точками или у границ. На левой панели диапазон y ограничен для читаемости: уходящие за него фрагменты не исчезли из предсказания. На правой панели виден рост валидационной ошибки при низкой обучающей — это проявление переобучения.

### 5.3. Ridge: добавим предпочтение небольших весов
В этом ноутбуке используем соглашение:
$$J_\lambda(w,b)=\frac1n\|Xw+b-y\|_2^2+\lambda\|w\|_2^2.$$
**Свободный коэффициент $b$ не штрафуем.** Масштабирование признаков особенно важно: штраф зависит от единиц измерения коэффициентов.

Для весов:
$$\nabla_w J_\lambda=\frac2nX^\top(Xw+b-y)+2\lambda w.$$
Для сдвига регуляризационная добавка равна нулю.

**Как штраф меняет решение.** Для центрированных данных без отдельного сдвига:
$$\nabla J_\lambda(w)=\frac2nX^\top(Xw-y)+2\lambda w=0,$$
$$X^\top Xw-X^\top y+n\lambda w=0,$$
$$(X^\top X+n\lambda I)w=X^\top y,$$
$$\hat w_\lambda=(X^\top X+n\lambda I)^{-1}X^\top y.$$
При $\lambda>0$ добавка к диагонали делает систему обратимой даже при зависимых столбцах $X$. Для GD вместо решения системы используется обновление
$$w_{t+1}=(1-2\eta\lambda)w_t-\eta\frac2nX^\top(Xw_t-y).$$
По сравнению с обычным GD появляется дополнительный вклад, направленный к нулю. Ниже отдельно учтём свободный коэффициент, который не штрафуется.

`sklearn.Ridge` минимизирует **сумму**, а не среднее квадратов, плюс `alpha` × квадрат нормы. Поэтому для совпадения с нашей формулой передаём **`alpha = n * lambda`**.

#### Полный вывод Ridge со свободным коэффициентом
Обозначим расширенную матрицу $A=[X\ \mathbf1]$, параметры $\theta=(w^\top,b)^\top$ и маску штрафа $P=\operatorname{diag}(1,\dots,1,0)$. Тогда
$$J_\lambda(\theta)=\frac1n\|A\theta-y\|^2+\lambda\theta^\top P\theta,$$
$$\nabla J_\lambda(\theta)=\frac2nA^\top(A\theta-y)+2\lambda P\theta.$$
Приравниваем к нулю:
$$(A^\top A+n\lambda P)\hat\theta=A^\top y.$$
При $\lambda>0$ и хотя бы одном объекте эта система имеет единственное решение: из
$$u^\top(A^\top A+n\lambda P)u=\|Au\|^2+n\lambda\|u_{1:d}\|^2$$
следует, что нулевое значение возможно только при нулевых весах и нулевом сдвиге.

**Условия важны:** это верно, когда единственная нештрафуемая координата — один столбец единиц. Если оставлять без штрафа несколько зависимых столбцов, единственность не гарантирована.

### 5.4. Ползунок регуляризации
Фиксируем степень полинома 15 и меняем только силу штрафа. Масштабирование обучается **только на train**, затем применяется к валидации.

Слева — предсказание, в центре — веса стандартизованных полиномиальных признаков, справа — ошибка. Суммарная норма весов Ridge уменьшается с усилением штрафа, но отдельный коэффициент не обязан меняться монотонно.

**Вопрос:** что произойдёт с предсказанием при очень большом штрафе, если сдвиг не штрафуется?

#### Формализация выбора гиперпараметра
Для каждого $\lambda$ из заранее заданной сетки $\Lambda$ обучаем модель **только на train**, затем считаем
$$\widehat R_{val}(\lambda)=\frac1{n_{val}}\sum_{i\in val}(f_{\hat\theta_\lambda}(x_i)-y_i)^2,
\qquad\hat\lambda\in\arg\min_{\lambda\in\Lambda}\widehat R_{val}(\lambda).$$

Валидация участвует в выборе $\hat\lambda$, поэтому минимум валидационной ошибки не является независимой оценкой качества выбранной модели. Для итоговой оценки нужен отдельный тест. На нём нельзя выбирать степень полинома, штраф или число шагов.

После выбора параметров можно переобучить модель на train + validation. В нашей нормировке $\lambda$ при этом остаётся тем же, а `Ridge.alpha = n * lambda` пересчитывается по новому размеру обучающей выборки.

In [ ]:
lambda_grid = np.logspace(-8, 2, 70)
ridge_train, ridge_valid = [], []
for lam in lambda_grid:
    model = polynomial_model(15, alpha=len(y_train_poly)*lam).fit(x_train_poly,y_train_poly)
    ridge_train.append(np.mean((model.predict(x_train_poly)-y_train_poly)**2))
    ridge_valid.append(np.mean((model.predict(x_valid_poly)-y_valid_poly)**2))

def regularization_demo(log_lambda=-3.):
    lam = 10.**log_lambda
    model = polynomial_model(15, alpha=len(y_train_poly)*lam).fit(x_train_poly,y_train_poly)
    coef = model[-1].coef_
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].scatter(x_train_poly,y_train_poly,s=22,c="black",label="Обучение")
    axes[0].plot(x_plot_poly,true_curve(x_plot_poly),c=GREEN,ls="--",label="Истина")
    axes[0].plot(x_plot_poly,model.predict(x_plot_poly),c=BLUE,label="Ridge")
    axes[0].set(xlabel="x",ylabel="y",ylim=(-2,2),title=f"λ={lam:.1e}"); axes[0].legend(fontsize=8)
    axes[1].bar(np.arange(1,16),coef,color=BLUE)
    axes[1].set(xlabel="Степень признака",ylabel="Коэффициент",title=f"Норма весов: {np.linalg.norm(coef):.2f}")
    axes[2].loglog(lambda_grid,ridge_train,label="Обучение")
    axes[2].loglog(lambda_grid,ridge_valid,label="Валидация")
    axes[2].axvline(lam,c=ORANGE,ls="--")
    axes[2].set(xlabel="λ",ylabel="MSE",title="Выбираем λ по валидации")
    axes[2].legend(fontsize=8); plt.tight_layout(); plt.show()

regularization_demo()
show_controls(regularization_demo, log_lambda=(-8.,2.,.25))
best_lambda = lambda_grid[np.argmin(ridge_valid)]
print(f"Лучшее λ в этой сетке по валидации: {best_lambda:.3g}")
print("Это оценка для выбора параметра, не независимая итоговая тестовая оценка.")

**Вывод эксперимента.** При усилении штрафа сложная модель становится менее гибкой; при очень большом штрафе веса приближаются к нулю, а нештрафуемый сдвиг — к среднему обучающих ответов. На этом наборе валидационная ошибка меняется немонотонно. Выбираем минимум среди проверенных λ, а не значение с самыми маленькими весами.

### 5.5. ★ Lasso: некоторые веса становятся нулевыми
L2 — сумма квадратов коэффициентов, L1 — сумма модулей. В форме ограничений допустимые области в двух измерениях — круг и ромб. У ромба есть вершины на осях: оптимум может попасть в точку с нулевым коэффициентом.

На графике показываем границу одного уровня штрафа и линии уровня ошибки. Геометрия иллюстрирует механизм, но не означает, что Lasso всегда выбирает нужные признаки или единственный из коррелирующих признаков.

**Две демонстрации.** Сначала рассматриваем одну и ту же квадратичную ошибку при ограничении нормы весов: красная точка показывает минимум внутри круга или ромба. Затем генерируем десять признаков, из которых только два влияют на ответ, и сравниваем коэффициенты Ridge и Lasso. Параметры штрафа фиксированы для иллюстрации, подбор качества здесь не выполняется.

#### ★ Почему L1 даёт точные нули: условие оптимальности
Зафиксируем соглашение
$$J_\lambda(w,b)=\frac1n\|Xw+b-y\|^2+\lambda\sum_j|w_j|.$$
Модуль не дифференцируем в нуле, но имеет субдифференциал:
$$\partial|u|=\{1\}\ (u>0),\quad\{-1\}\ (u<0),\quad[-1,1]\ (u=0).$$
Для оптимального решения необходимо и достаточно $0\in\partial J_\lambda$ (задача выпукла). Для координаты $j$:
$$0\in\frac2n x_j^\top(Xw+b-y)+\lambda\partial|w_j|.$$
Поэтому нулевая координата совместима с оптимальностью, когда
$$\left|\frac2n x_j^\top(y-Xw-b)\right|\le\lambda.$$
L2 такого интервала в нуле не имеет: производная штрафа равна $2\lambda w_j$.

При ортонормированных столбцах $X^\top X/n=I$ и центрированных данных задача распадается по координатам. Если $z=X^\top y/n$, то
$$\hat w_j=\operatorname{sign}(z_j)\max(|z_j|-\lambda/2,0).$$
Это операция **мягкого порога**. Для произвольных признаков приведённая формула уже не является общим решением.

`sklearn.Lasso` использует $\|Xw+b-y\|^2/(2n)+\texttt{alpha}\|w\|_1$. Для совпадения с нашим соглашением нужно `alpha = lambda / 2`. Это другая нормировка, чем у `Ridge`.

In [ ]:
u = np.linspace(-1.4,2.5,220)
v = np.linspace(-1.4,1.5,220)
U,V = np.meshgrid(u,v)
Z = (U-2.)**2 + (V-.4)**2
fig, axes = plt.subplots(1,2,figsize=(11,4))
angles = np.linspace(0,2*np.pi,250)
for ax in axes:
    ax.contour(U,V,Z,levels=[1.,1.16,1.6,2.2,3.,4.5,6.],cmap="Blues")
    ax.scatter(2,.4,marker="*",s=120,c=ORANGE,label="Минимум без ограничения")
    ax.axhline(0,c="gray",lw=.5); ax.axvline(0,c="gray",lw=.5)
    ax.set(xlabel="w₁",ylabel="w₂",aspect="equal",xlim=(-1.4,2.5),ylim=(-1.4,1.5))
axes[0].fill(np.cos(angles),np.sin(angles),color=GREEN,alpha=.2)
axes[0].plot(np.cos(angles),np.sin(angles),c=GREEN)
projection = np.array([2.,.4])/np.linalg.norm([2.,.4])
axes[0].scatter(*projection,c=RED,s=60); axes[0].set_title("L2: w₁² + w₂² ≤ 1")
axes[1].fill([1,0,-1,0],[0,1,0,-1],color=GREEN,alpha=.2)
axes[1].plot([1,0,-1,0,1],[0,1,0,-1,0],c=GREEN)
axes[1].scatter(1,0,c=RED,s=60); axes[1].set_title("L1: |w₁| + |w₂| ≤ 1; здесь w₂=0")
plt.tight_layout(); plt.show()

rng_sparse = np.random.default_rng(SEED)
X_sparse = rng_sparse.normal(size=(120,10))
y_sparse = X_sparse@np.r_[3.,-2.,np.zeros(8)] + rng_sparse.normal(0,.5,120)
Xs = StandardScaler().fit_transform(X_sparse)
ridge_sparse = Ridge(alpha=12).fit(Xs,y_sparse)
lasso_sparse = Lasso(alpha=.1,max_iter=10000).fit(Xs,y_sparse)
fig, ax = plt.subplots(figsize=(10,3))
ids=np.arange(10)
ax.bar(ids-.18,ridge_sparse.coef_,width=.36,label="Ridge",color=BLUE)
ax.bar(ids+.18,lasso_sparse.coef_,width=.36,label="Lasso",color=ORANGE)
ax.set(xlabel="Номер признака (полезны только 0 и 1)",ylabel="Вес",title="Пример разреженного решения")
ax.set_xticks(ids); ax.legend(); plt.show()
print("Число практически нулевых весов Lasso:", np.sum(np.abs(lasso_sparse.coef_)<1e-8))
print("Параметры выбраны для иллюстрации; одинаковые alpha у Ridge и Lasso не означают одинаковый штраф.")

**Как связаны рисунки.** В геометрическом примере минимум на ромбе попал на ось, поэтому один вес точно равен нулю. В примере с десятью признаками Lasso тоже обнуляет часть коэффициентов, тогда как Ridge обычно лишь уменьшает их. Ненулевой вес не является доказательством причинного влияния признака.

## 6. Модель, критерий и алгоритм обучения
### 6.1. Четыре разные сущности
| Что | В нашем примере |
|---|---|
| Модель | $\hat y=Xw$ |
| Критерий обучения | MSE или MSE + L2 |
| Алгоритм оптимизации | Прямое решение, GD, SGD |
| Гиперпараметры | Шаг $\eta$, штраф $\lambda$, число шагов |

### 6.2. Структура заданий
Графики, генерация данных и код проверок уже готовы. Ваша задача — вычисления градиента и эксперимент. Заполняйте только ячейки, помеченные **ВАШ КОД**.

В задании 7.1 реализуем обучение, в 7.2 исследуем шаг, в 7.3 формулируем выводы. Задание 7.4 расширяет реализацию L2-регуляризацией.

## 7. Задания
### 7.1. Реализуйте MSE, градиент и обучение
В этой части $X$ уже содержит **последний столбец единиц**. Последний элемент $w$ — свободный коэффициент. Массивы `y` и `w` одномерные.

1. Допишите `student_mse` и `student_gradient` без циклов по объектам.
2. В `student_fit` допишите только обновление весов.
3. Запустите проверки и посмотрите график.

Ожидается, что веса приблизятся к решению `lstsq`. **Возвращаемые значения:** итоговые веса и массив MSE длины `steps + 1`, включая ошибку до первого обновления.

In [ ]:
def student_mse(X, y, w):
    # ВАШ КОД: вернуть средний квадрат остатка (число).
    raise NotImplementedError("Допишите student_mse")

def student_gradient(X, y, w):
    # ВАШ КОД: вернуть градиент, массив той же формы, что w.
    raise NotImplementedError("Допишите student_gradient")

def student_fit(X, y, lr=.1, steps=200):
    w = np.zeros(X.shape[1], dtype=float)
    losses = [student_mse(X, y, w)]
    for _ in range(steps):
        # ВАШ КОД: один шаг обновления w с помощью student_gradient.
        raise NotImplementedError("Допишите обновление весов в student_fit")
        losses.append(student_mse(X, y, w))
    return w, np.asarray(losses)

In [ ]:
def check_student_part1():
    X_check = np.array([[1., 0., 1.], [0., 2., 1.], [-1., 1., 1.]])
    y_check = np.array([1., -2., .5])
    w_check = np.array([.3, -.4, .2])
    expected = np.mean((X_check@w_check-y_check)**2)
    np.testing.assert_allclose(student_mse(X_check,y_check,w_check), expected)
    g = np.asarray(student_gradient(X_check,y_check,w_check))
    assert g.shape == w_check.shape, "Проверьте размерность градиента"
    # Независимая проверка производной центральной конечной разностью.
    eps = 1e-6
    numeric = np.array([(student_mse(X_check,y_check,w_check+eps*e)-
                         student_mse(X_check,y_check,w_check-eps*e))/(2*eps)
                        for e in np.eye(len(w_check))])
    np.testing.assert_allclose(g,numeric,rtol=1e-5,atol=1e-6)
    X = np.c_[x_line,np.ones(len(x_line))]
    fitted, losses = student_fit(X,y_line,lr=.1,steps=200)
    np.testing.assert_allclose(fitted,np.linalg.lstsq(X,y_line,rcond=None)[0],atol=1e-4)
    assert len(losses)==201, "История должна включать начальную ошибку"
    assert losses[-1] < losses[0], "Ошибка не уменьшилась"
    np.testing.assert_allclose(losses[-1],student_mse(X,y_line,fitted),rtol=1e-7)
    fig,ax=plt.subplots()
    ax.semilogy(losses)
    ax.set(xlabel="Шаг",ylabel="MSE",title="Ваша реализация обучается")
    plt.show()
    print("Все проверки пройдены. Веса:",fitted.round(4))

try:
    check_student_part1()
except NotImplementedError as error:
    print("Задание ещё не заполнено:", error)

### 7.2. Исследуйте шаг обучения
Запустите вашу реализацию с тремя шагами: медленное обучение, быстрая сходимость, расходимость. Начальный набор — `[0.005, 0.1, 0.9]`; можно предложить свой.

График строится автоматически. Расходящийся запуск ограничиваем 40 шагами.

**Запишите под графиком:**
1. Какой шаг оказался лучшим при данном бюджете и почему?
2. Почему растущая ошибка не означает, что линейная модель не подходит данным?
3. Что будет с градиентом и подходящим шагом, если вместо MSE использовать сумму квадратов?

In [ ]:
LEARNING_RATES = [.005, .1, .9]  # ВАШ ЭКСПЕРИМЕНТ: меняйте эти значения.
try:
    results_student = [(lr, student_fit(X_line,y_line,lr=lr,steps=40)[1]) for lr in LEARNING_RATES]
    fig,ax=plt.subplots()
    for lr,losses in results_student:
        ax.semilogy(losses,label=f"η={lr}")
    ax.set(xlabel="Шаг",ylabel="MSE",title="Ваш эксперимент с шагом")
    ax.legend(); plt.show()
except NotImplementedError:
    print("Сначала завершите задание 7.1.")

**Ваши наблюдения:**

1. …
2. …
3. …

### 7.3. Сравнение шага и регуляризации
Объясните, чем изменение шага отличается от изменения силы регуляризации: что меняет путь к решению, а что — сам критерий и его минимум?

### 7.4. ★ Добавьте L2-регуляризацию
Реализуйте градиент
$$J_\lambda(w)=\operatorname{MSE}(Xw,y)+\lambda\sum_{j=1}^{p-1}w_j^2.$$
Последний вес — сдвиг, его **не штрафуем**. Не изменяйте входной `w` на месте.

Проверка сравнит градиент с численной производной, а обучение — с `Ridge(alpha=n*lambda)`. Затем сравнит нестабильность коэффициентов при коррелирующих признаках.

In [ ]:
def student_ridge_gradient(X, y, w, lam):
    # ВАШ КОД: градиент MSE + штраф только для w[:-1].
    raise NotImplementedError("Допишите student_ridge_gradient")

In [ ]:
def check_student_ridge():
    X = np.array([[1.,2.,1.],[-1.,0.,1.],[2.,-1.,1.],[0.,1.,1.]])
    y = np.array([2.,-1.,3.,.5])
    w = np.array([.3,-.2,.7]); before=w.copy(); lam=.2
    def objective(v): return np.mean((X@v-y)**2)+lam*np.sum(v[:-1]**2)
    eps=1e-6
    numeric=np.array([(objective(w+eps*e)-objective(w-eps*e))/(2*eps) for e in np.eye(3)])
    actual=student_ridge_gradient(X,y,w,lam)
    np.testing.assert_array_equal(w,before,err_msg="Не изменяйте входной w")
    np.testing.assert_allclose(actual,numeric,atol=1e-6)
    fitted=np.zeros(3)
    for _ in range(1200): fitted-=.05*student_ridge_gradient(X,y,fitted,lam)
    reference=Ridge(alpha=len(y)*lam).fit(X[:,:-1],y)
    np.testing.assert_allclose(fitted,np.r_[reference.coef_,reference.intercept_],atol=1e-5)
    print("Градиент и обучение Ridge проверены; сдвиг не штрафуется.")
    local_rng=np.random.default_rng(SEED)
    X_aug=np.c_[X_col,np.ones(len(X_col))]
    weights={0.:[],.1:[]}
    for _ in range(25):
        target=y_col_true+local_rng.normal(0,.3,len(X_col))
        for strength in weights:
            w=np.zeros(3)
            # Достаточно итераций, чтобы МНК успел проявить неустойчивость.
            if strength==0:
                w=np.linalg.lstsq(X_aug,target,rcond=None)[0]
            else:
                for step in range(1000):
                    w-=.1*student_ridge_gradient(X_aug,target,w,strength)
            weights[strength].append(w.copy())
    fig,ax=plt.subplots()
    for strength, values in weights.items():
        values=np.asarray(values)
        ax.scatter(values[:,0],values[:,1],label=f"λ={strength}")
    ax.set(xlabel="w₁",ylabel="w₂",title="Изменяем ответы: как устойчивы ваши веса?")
    ax.legend(); plt.show()

try:
    check_student_ridge()
except NotImplementedError as error:
    print("Дополнительное задание ещё не заполнено:",error)

## 8. Основные выводы
- MSE соответствует максимальному правдоподобию при независимом нормальном шуме с общей дисперсией.
- Линейная регрессия — модель; градиентный спуск — один из способов её обучить.
- Прямое решение удобно, но его стоимость растёт с размерностью; GD тоже требует времени до сходимости.
- Шаг и масштаб признаков влияют на траекторию обучения.
- Регуляризация меняет критерий: допускает смещение решения ради уменьшения его чувствительности к данным.
- Гиперпараметры выбирают по валидации. Финальное качество оценивают на отдельном тесте.

### Дальнейшее чтение
- [Учебник Stanford CS229: линейная регрессия и вероятностная интерпретация](https://cs229.stanford.edu/notes2022fall/main_notes.pdf)
- [NumPy: numpy.linalg.lstsq](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
- [scikit-learn: линейные модели, Ridge и Lasso](https://scikit-learn.org/stable/modules/linear_model.html)
- [Орельен Жерон: Training Models](https://github.com/ageron/handson-ml3/blob/main/04_training_linear_models.ipynb)

Все данные, графики и анимация в этом ноутбуке генерируются локально; сторонние изображения не используются.